In [1]:
import requests
import matplotlib.pyplot as plt
import numpy as np

In [2]:
# url = "http://127.0.0.1:8000/"
url = "https://gd-secret-api.onrender.com/"

evaluate_url = url + "evaluate"
evaluate_with_grad_url = url + "evaluate-with-gradient"
discovery_url = url + "problems"


In [3]:

def _is_json_like(obj):
    return isinstance(obj, (dict, list))


def test_evaluate_default_x_returns_json():
    resp = requests.post(
        evaluate_url,
        json={"problem_id": "parabola_1d", "x": [2.5]},
    )
    assert resp.status_code == 200
    data = resp.json()
    assert _is_json_like(data)


In [4]:
import requests
import pandas as pd

# API endpoints
# base_url = "https://gd-secret-api.onrender.com"
base_url = url.rstrip("/")

evaluate_url = f"{base_url}/evaluate"
gradient_url = f"{base_url}/evaluate-with-gradient"
problems_url = f"{base_url}/problems"


# 1. Ask the server: "Which problems do you have?"
print(f'Contacting endpoint {problems_url}')
problems = requests.get(problems_url).json()

print("Available problems:")
for problem_id, info in problems.items():
    # print(f"- {problem_id}: info{info}")
    print(f"- {problem_id}: {info['name']} ({info['dimension']}D)")
    print(f"  Description:{info['description']}")
    print(f"  Input: {info['input_description']}")
    


Contacting endpoint https://gd-secret-api.onrender.com/problems
Available problems:
- parabola_1d: Simple 1D parabola (1D)
  Description:Easy convex function. Good first gradient descent example.


KeyError: 'input_description'

In [ ]:
# 2. Evaluate one point on a simple 1D function
response = requests.post(
    evaluate_url,
    json={
        "problem_id": "parabola_1d",
        "x": [2.5]
    }
)

data = response.json()

print("\nSingle API call:")
print(data)



Single API call:
{'problem_id': 'parabola_1d', 'x': [2.5], 'y': 0.25}


In [ ]:
# 3. Try several x values and collect the answers
rows = []

for x_value in [-1, 0, 1, 2, 2.5, 3, 4, 5]:
    response = requests.post(
        evaluate_url,
        json={
            "problem_id": "parabola_1d",
            "x": [x_value]
        }
    )

    data = response.json()

    rows.append({
        "x": x_value,
        "y = f(x)": data["y"]
    })

df = pd.DataFrame(rows)

print("\nEvaluating the function at several points:")
display(df)


# 4. Ask the API for the gradient too
response = requests.post(
    gradient_url,
    json={
        "problem_id": "parabola_1d",
        "x": [2.5]
    }
)

data = response.json()

print("\nFunction value + gradient:")
print(data)

print(
    f"\nAt x = {data['x'][0]}, "
    f"the function value is {data['y']}, "
    f"and the gradient is {data['gradient'][0]}."
)


Evaluating the function at several points:


,x,y = f(x)
0,-1.0,16.00
1,0.0,9.00
2,1.0,4.00
3,2.0,1.00
4,2.5,0.25
5,3.0,0.00
6,4.0,1.00
7,5.0,4.00



Function value + gradient:
{'problem_id': 'parabola_1d', 'x': [2.5], 'y': 0.25, 'gradient': [-1.0]}

At x = 2.5, the function value is 0.25, and the gradient is -1.0.


In [ ]:
def test_evaluate_parabola_1d_returns_json():
    resp = requests.post(
        evaluate_url,
        json={
            "problem_id": "parabola_1d",
            "x": [2.5]
        },
    )
    assert resp.status_code == 200
    data = resp.json()
    assert _is_json_like(data)


In [ ]:
def evaluate_1d(x):
    if isinstance(x, (int, float)):
        x = [x]
    if isinstance(x, np.ndarray):
        x = x.tolist()
    resp = requests.post(
        evaluate_url,
        json={
            "problem_id": "parabola_1d",
            "x": x
        },
    )
    assert resp.status_code == 200
    data = resp.json()
    assert _is_json_like(data)
    return data["y"]


def evaluate_1d_calc_grad(x, eps=1e-6):
    if isinstance(x, (int, float)):
        x = np.array([x])
    x_plus_eps = (x + eps).tolist()
    x_minus_eps = (x - eps).tolist()
    # convert x_plus_eps and x_minus_eps to lists for the API
    y_plus = evaluate_1d(x_plus_eps)
    y_minus = evaluate_1d(x_minus_eps)
    y = evaluate_1d(x.tolist())
    grad = (y_plus - y_minus) / (2 * eps)
    return y, grad


def evaluate_2d(x_vec):
    """Evaluate a 2D problem by POSTing to the evaluate endpoint."""
    resp = requests.post(
        evaluate_url,
        json={
            "problem_id": "bowl_2d",
            "x": x_vec,
        },
    )
    assert resp.status_code == 200
    data = resp.json()
    assert _is_json_like(data)
    return data["y"]

def evaluate_calc_grad(x_vec, api_func, eps=1e-6):
    """Compute numerical gradient for a 2D input using central differences."""
    # ensure we have a mutable copy
    grad = np.zeros_like(x_vec)
    for i in range(x_vec.shape[0]):
        x_plus = list(x_vec)
        x_minus = list(x_vec)
        x_plus[i] += eps
        x_minus[i] -= eps
        y_plus = api_func(x_plus)
        y_minus = api_func(x_minus)
        grad[i] = (y_plus - y_minus) / (2 * eps)
    y = api_func(x_vec)
    return y, grad

# print(evaluate_1d(2.5))
# print(evaluate_1d_calc_grad(2.5))
# print(evaluate_2d([10.0, -3.0]))
# print(evaluate_2d_calc_grad([10.0, -3.0]))

In [ ]:
curr_diff = np.inf
curr_x = np.array([np.random.rand() * 10])
step_size = 0.01


print(f"Initial x: {curr_x}")
print(f"Initial y: {evaluate_1d(curr_x)}")
print(f"Step size: {step_size}")

y_values = []
grad_sizes = []

while curr_diff > 1e-5:
    # y, grad = evaluate_1d_calc_grad(curr_x)
    y, grad = evaluate_calc_grad(curr_x, evaluate_1d)
    y_values.append(y)
    grad = np.array(grad)
    grad_sizes.append(np.linalg.norm(grad))
    print(f"x: {curr_x}, y: {y}, grad: {grad}")
    next_x = curr_x - step_size * grad
    curr_diff = abs(next_x - curr_x)
    curr_x = next_x

print(f"Final x: {curr_x}, Final y: {evaluate_1d(curr_x)}")

Initial x: [4.39049348]
Initial y: 1.9334721080991812
Step size: 0.01
x: [4.39049348], y: 1.9334721080991812, grad: [2.78098695]
x: [4.36268361], y: 1.8569066126091458, grad: [2.72536721]
x: [4.33542993], y: 1.783373110739931, grad: [2.67085987]
x: [4.30872134], y: 1.7127515355454583, grad: [2.61744267]
x: [4.28254691], y: 1.644926574730009, grad: [2.56509382]
x: [4.25689597], y: 1.5797874823608318, grad: [2.51379194]
x: [4.23175805], y: 1.517227898050437, grad: [2.4635161]
x: [4.20712289], y: 1.457145673278215, grad: [2.41424578]
x: [4.18298043], y: 1.3994427046090006, grad: [2.36596087]
x: [4.15932082], y: 1.344024773498926, grad: [2.31864165]
x: [4.13613441], y: 1.2908013924609612, grad: [2.27226882]
x: [4.11341172], y: 1.2396856573131876, grad: [2.22682344]
x: [4.09114349], y: 1.1905941052781677, grad: [2.18228697]
x: [4.06932062], y: 1.143446578702703, grad: [2.13864123]
x: [4.0479342], y: 1.0981660941785458, grad: [2.09586841]
x: [4.02697552], y: 1.0546787168414677, grad: [2.0539

In [ ]:
# plot y_values over time
plt.plot(y_values)
plt.plot(grad_sizes)
plt.xlabel("Iteration")
plt.ylabel("y value") 